#Debugged and quoted Code 

In [ ]:
"""Healthcare Professional Analytics Pipeline - FULLY FIXED VERSION"""  # Module docstring: describes pipeline purpose; not used as a variable.
"Complete system for scoring and clustering doctors based on 4 personality buckets"  # Additional description (string literal, no variable assignment).
"Handles encoding issues, missing columns, and digital data processing"  # Additional description.

"Author: Subhoit Talukdar"  
"Version: Beta 1.3 - With debug function for social media analysis"

import pandas as pd  # Import pandas as pd — used throughout for DataFrame operations (load, merge, groupby, to_csv).
import numpy as np  # Import numpy as np — used for numerical ops (np.log10, np.isnan, np.nan_to_num).
from sklearn.cluster import KMeans  # Import KMeans — used in perform_clustering() to compute clusters.
from sklearn.preprocessing import StandardScaler  # Import StandardScaler — used in perform_clustering() to scale features.
from sklearn.metrics import silhouette_score  # Import silhouette_score — used to evaluate clustering quality in perform_clustering().
import warnings  # Import warnings — used to suppress specific warnings via filterwarnings().
import chardet  # Import chardet — used in detect_encoding() to auto-detect file encodings.
warnings.filterwarnings('ignore', category=FutureWarning, message='.*frequency.*')  # Suppress FutureWarnings matching message; improves console output clarity.

# =====================================================
# CONFIGURATION
# =====================================================

# File paths
DATA_DIR = r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\Model_Data'  # DATA_DIR: base folder where all CSVs live; used when building file_path in load functions.

# Sample file names
FILES = {  # FILES: dict storing file names keyed by logical dataset names; used by load_csv_safe() and load_engagement_data().
    # Engagement data (Jan-July 2025)
    'engagement': [  # 'engagement' key maps to list of monthly CSV names — used in load_engagement_data().
        'Jan25.csv', 'Feb25.csv', 'March25.csv',
        'April25.csv', 'May25.csv', 'June25.csv', 'July25.csv'
    ],
    
    # Professional info
    'pi': 'pi.csv',  # 'pi' key -> used as master list in aggregate_data_by_uin()
    'clinics': 'clinics.csv',  # 'clinics' -> used in aggregate_data_by_uin()
    
    # Academic/Research data
    'publications': 'Publications.csv',  # 'publications' used in aggregate_data_by_uin()
    'trials': 'Trials.csv',  # 'trials' used in aggregate_data_by_uin()
    'academic_associations': 'Academic_association.csv',  # used in aggregate_data_by_uin()
    
    # Social Media data
    'digital': 'digital.csv',  # 'digital' used in debug_digital_data() and aggregate_data_by_uin()
    'healthcare_platforms': 'Healthcare.csv',  # used in aggregate_data_by_uin()
    
    # Recognition data
    'awards': 'awards.csv',  # used in aggregate_data_by_uin()
    'press': 'press.csv',  # used in aggregate_data_by_uin()
    'events': 'events.csv',  # used in aggregate_data_by_uin()
    'associations': 'association.csv'  # used in aggregate_data_by_uin()
}

# Clustering eligibility rules
ELIGIBILITY_MODE = 'basic'  # 'strict', 'moderate', or 'relaxed' — referenced by assess_clustering_eligibility() to choose rules.

ELIGIBILITY_RULES = {  # ELIGIBILITY_RULES: dict of rules referenced by assess_clustering_eligibility()
    'strict': {'min_buckets': 4, 'engagement_required': False},
    'moderate': {'min_buckets': 3, 'engagement_required': False},
    'relaxed': {'min_buckets': 2, 'engagement_required': False},
    'basic': {'min_buckets': 1, 'engagement_required': False}
}

# =====================================================
# ENHANCED SCORING CONFIGURATION - ALL 4 BUCKETS
# =====================================================

SCORING_CONFIG = {
    # ===== BUCKET 1: ENGAGEMENT =====
    'engagement': {
        # Email component weights
        'email_open_weight': 0.50,
        'email_click_weight': 0.50,
        
        # WhatsApp component weights
        'whatsapp_read_weight': 0.50,
        'whatsapp_click_weight': 0.50,
        
        # Call component weights
        'call_productive_weight': 0.70,
        'call_duration_weight': 0.30,
        
        # Final engagement score composition
        'final_email_weight': 0.33,
        'final_whatsapp_weight': 0.33,
        'final_call_weight': 0.34,
        
        # Max score cap (optional, default 100)
        'max_score': 100
    },
    
    # ===== BUCKET 2: ACADEMIC/RESEARCH =====
    'academic': {
        # Publication scoring
        'publication_points_per_item': 5,      # Points per publication
        'publication_max_score': 40,           # Cap for publications component
        
        # Clinical trials scoring
        'trial_points_per_item': 15,           # Points per trial (higher weight)
        'trial_max_score': 30,                 # Cap for trials component
        
        # Academic associations scoring
        'association_points_per_item': 10,     # Points per academic association
        'association_max_score': 30,           # Cap for associations component
        
        # Component weights (how much each contributes to final academic score)
        'publication_weight': 0.40,            # 40% from publications
        'trial_weight': 0.30,                  # 30% from trials
        'association_weight': 0.30,            # 30% from associations
        
        # Alternative: If you want simple addition (current method)
        'use_weighted_composition': False,     # Set to True to use weights above
        
        # Max score cap
        'max_score': 100
    },
    
    # ===== BUCKET 3: SOCIAL MEDIA INCLINATION =====
    'social_media': {
        # Follower count scoring (logarithmic scale)
        'follower_log_multiplier': 10,         # log10(followers) * 10
        'follower_max_score': 50,              # Cap for follower component
        'follower_min_threshold': 100,         # Min followers to count (avoid noise)
        
        # Platform count scoring
        'platform_points_per_item': 10,        # Points per social platform
        'platform_max_score': 30,              # Cap for platforms component
        
        # Healthcare platform scoring
        'healthcare_platform_points_per_item': 10,  # Points per healthcare platform
        'healthcare_platform_max_score': 20,        # Cap for healthcare platforms
        
        # Component weights
        'follower_weight': 0.50,               # 50% from followers
        'platform_weight': 0.30,               # 30% from general platforms
        'healthcare_platform_weight': 0.20,    # 20% from healthcare platforms
        
        # Alternative: Simple addition (current method)
        'use_weighted_composition': False,
        
        # Max score cap
        'max_score': 100
    },
    
    # ===== BUCKET 4: RECOGNITION =====
    'recognition': {
        # Awards scoring
        'award_points_per_item': 15,           # Points per award
        'award_max_score': 30,                 # Cap for awards component
        
        # Press mentions scoring
        'press_points_per_item': 10,           # Points per press mention
        'press_max_score': 25,                 # Cap for press component
        
        # Event participation scoring
        'event_points_per_item': 8,            # Points per event
        'event_max_score': 25,                 # Cap for events component
        
        # Professional association scoring
        'association_points_per_item': 5,      # Points per association
        'association_max_score': 20,           # Cap for associations component
        
        # Component weights
        'award_weight': 0.30,                  # 30% from awards
        'press_weight': 0.25,                  # 25% from press
        'event_weight': 0.25,                  # 25% from events
        'association_weight': 0.20,            # 20% from associations
        
        # Alternative: Simple addition (current method)
        'use_weighted_composition': False,
        
        # Max score cap
        'max_score': 100
    }
}


# =====================================================
# ENHANCED FILE LOADING WITH ENCODING DETECTION
# =====================================================

def detect_encoding(file_path):  # Function detect_encoding(file_path) defined here; called by load_csv_with_encoding_detection().
    """Detect file encoding using chardet"""  # Docstring explaining purpose.
    try:
        with open(file_path, 'rb') as file:  # Open file in binary mode to sample bytes for chardet; file_path built using DATA_DIR + FILES elsewhere.
            raw_data = file.read(100000)  # Read up to first 100k bytes as sample for detection; local var raw_data.
            result = chardet.detect(raw_data)  # Use chardet to guess encoding; result is a dict with 'encoding' and 'confidence'.
            encoding = result['encoding']  # encoding variable holds detected encoding string; returned to caller.
            confidence = result['confidence']  # confidence float for logging.
            print(f"      Detected encoding: {encoding} (confidence: {confidence:.2f})")  # Console log for debugging.
            return encoding  # Return encoding to load_csv_with_encoding_detection().
    except Exception as e:
        print(f"      Could not detect encoding: {e}")  # Log failure to detect encoding.
        return None  # Return None if detection fails.

def load_csv_with_encoding_detection(file_path):  # Function to robustly load CSVs with fallback encodings; used by load_csv_safe() and load_engagement_data().
    """Try to load CSV with multiple encoding strategies"""  # Docstring.
    encodings_to_try = []  # local list to accumulate encodings to attempt.
    
    detected = detect_encoding(file_path)  # Call detect_encoding() above; result used first if present.
    if detected:
        encodings_to_try.append(detected)  # Add detected encoding first to try.

    encodings_to_try.extend(['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'windows-1252'])  # Add common fallbacks.

    for encoding in encodings_to_try:  # Try each encoding until read succeeds.
        try:
            df = pd.read_csv(file_path, encoding=encoding, low_memory=False, on_bad_lines='skip')  # Attempt read with this encoding.
            print(f"      Successfully loaded with {encoding} encoding")  # Log which encoding worked.
            return df  # Return DataFrame to caller (load_csv_safe or load_engagement_data).
        except (UnicodeDecodeError, LookupError):
            continue  # If decoding or lookup error, try next encoding silently.
        except Exception as e:
            print(f"      Error with {encoding}: {e}")  # Log any other error, continue trying.
            continue

    try:
        df = pd.read_csv(file_path, encoding='utf-8', errors='ignore', low_memory=False, on_bad_lines='skip')  # Final fallback: read with errors ignored.
        print(f"      Loaded with UTF-8 (ignoring errors)")  # Log fallback success.
        return df  # Return DataFrame.
    except Exception as e:
        print(f"      Failed to load file: {e}")  # If final fallback fails, log and return empty DF.
        return pd.DataFrame()  # Return empty DataFrame to signal failure to caller.

# =====================================================
# STEP 1: DATA LOADING & MERGING
# =====================================================

def load_engagement_data():  # Function to load monthly engagement CSVs, standardize columns, and compute averages; returns engagement_avg DataFrame used in aggregate_data_by_uin().
    """Load and average engagement data from Jan-July"""  # Docstring explains return.
    print("\n📊 Loading Engagement Data (Jan-July 2025)...")  # Console log.

    engagement_dfs = []  # list to collect monthly DataFrames.
    total_rows_loaded = 0  # counter for diagnostics.

    for file in FILES['engagement']:  # Iterate filenames specified in FILES['engagement'].
        try:
            file_path = f"{DATA_DIR}\\{file}"  # Build full file path using DATA_DIR; consistent across load functions.
            print(f"   Loading {file}...")  # Log which file is being loaded.
            df = load_csv_with_encoding_detection(file_path)  # Use robust loader; df is DataFrame or empty DF.
            if not df.empty:
                engagement_dfs.append(df)  # Append loaded DataFrame to list.
                total_rows_loaded += len(df)  # Increment row count for diagnostics.
                print(f"   ✓ Loaded {file}: {len(df)} rows, {df['UIN'].nunique()} unique UINs")  # Log rows and unique UINs — expects 'UIN' column to exist in file.
            else:
                print(f"   ⚠ File is empty: {file}")  # Warn if df is empty.
        except FileNotFoundError:
            print(f"   ⚠ File not found: {file}")  # File missing — non-fatal.
        except Exception as e:
            print(f"   ❌ Error loading {file}: {e}")  # Any other exception logged.

    if not engagement_dfs:
        print("   ❌ No engagement data loaded!")  # If no monthly data read — return empty DF to caller.
        return pd.DataFrame()

    print(f"\n   📊 Total rows before combining: {total_rows_loaded}")  # Diagnostic print.
    combined = pd.concat(engagement_dfs, ignore_index=True)  # Combine monthly DataFrames into one DataFrame 'combined'.
    print(f"   📊 Combined dataframe: {len(combined)} rows")  # Print combined size.
    print(f"   📊 Unique UINs in combined: {combined['UIN'].nunique()}")  # Print unique UIN count — relies on 'UIN' present in combined.

    # CRITICAL FIX: Standardize column names (remove spaces, convert to lowercase with underscores)
    # BUT preserve UIN as uppercase
    column_mapping = {}  # dict to map original column names to standardized names.
    for col in combined.columns:
        if col == 'UIN':
            column_mapping[col] = 'UIN'  # Keep 'UIN' uppercase so merges later match expected casing.
        else:
            column_mapping[col] = col.strip().lower().replace(' ', '_')  # Normalize other columns to snake_case lowercase.

    combined = combined.rename(columns=column_mapping)  # Apply renaming to combined DF.
    print(f"   🔍 DEBUG - Columns after standardization: {combined.columns.tolist()}")  # Print columns for debug.

    expected_cols = {  # expected_cols is a mapping of engagement columns to default values; used to ensure consistent columns exist.
        'hcp_email_open_rate': 0,
        'hcp_email_click_rate': 0,
        'hcp_whatsapp_read_rate': 0,
        'hcp_whatsapp_click_rate': 0,
        'hcp_call_productive_rate': 0,
        'average_duration_connected_calls': 0
    }

    for col, default in expected_cols.items():  # Ensure each expected column exists and is numeric.
        if col not in combined.columns:
            combined[col] = default  # Create missing column with default 0.
            print(f"   ⚠ Column '{col}' not found, using default value: {default}")  # Log missing col.
        else:
            # Convert to numeric, handling any non-numeric values
            combined[col] = pd.to_numeric(combined[col], errors='coerce').fillna(default)  # Coerce non-numeric to NaN then fill with default.

    agg_dict = {col: 'mean' for col in expected_cols.keys() if col in combined.columns}  # Build aggregation dict (mean) for existing expected cols.
    engagement_avg = combined.groupby('UIN', as_index=False).agg(agg_dict)  # Group by 'UIN' and compute mean of each metric -> engagement_avg DF.

    print(f"   ✓ Averaged engagement data: {len(engagement_avg)} unique doctors")  # Log result rows.
    print(f"   🔍 DEBUG - UIN sample after averaging: {engagement_avg['UIN'].head(10).tolist()}")  # Sample UINs.

    return engagement_avg  # Return aggregated engagement DF (used later in aggregate_data_by_uin()).

def load_csv_safe(file_key):  # Helper to load non-engagement CSV files safely; used by load_all_data().
    """Safely load a CSV file with encoding detection"""
    filename = FILES[file_key]  # Lookup filename from FILES dict by key (e.g., 'pi', 'digital').
    file_path = f"{DATA_DIR}\\{filename}"  # Build full path.

    try:
        print(f"   Loading {file_key}...")  # Log.
        df = load_csv_with_encoding_detection(file_path)  # Attempt robust load; returns DataFrame or empty DF.
        if not df.empty and 'UIN' in df.columns:
            print(f"   ✓ {file_key}: {len(df)} rows, {len(df['UIN'].unique())} unique UINs")  # Success log — requires 'UIN' column.
            return df  # Return DF to caller.
        elif df.empty:
            print(f"   ⚠ {file_key} is empty")  # Warn when file loads but is empty.
            return pd.DataFrame()  # Return empty DF consistently.
        else:
            print(f"   ⚠ {file_key} loaded but missing 'UIN' column")  # If 'UIN' missing, treat as unusable.
            return pd.DataFrame()
    except FileNotFoundError:
        print(f"   ⚠ {file_key} not found: {filename}")  # File missing.
        return pd.DataFrame()
    except Exception as e:
        print(f"   ❌ Error loading {file_key}: {e}")  # Generic exception handling.
        return pd.DataFrame()

def load_all_data():  # Orchestrates loading of all datasets into 'data' dict; called from main().
    """Load all CSV files"""
    print("\n📁 Loading All Data Files...")  # Log.

    data = {}  # Dictionary that will hold loaded datasets keyed by logical names (engagement, pi, etc.).

    data['engagement'] = load_engagement_data()  # Load engagement monthly average DF — function defined above.
    data['pi'] = load_csv_safe('pi')  # Load PI data — used as master UIN list in aggregation.
    data['clinics'] = load_csv_safe('clinics')  # Load clinics data.
    data['publications'] = load_csv_safe('publications')  # Load publications.
    data['trials'] = load_csv_safe('trials')  # Load clinical trials.
    data['academic_associations'] = load_csv_safe('academic_associations')  # Load academic associations.
    data['digital'] = load_csv_safe('digital')  # Load digital/social data.
    data['healthcare_platforms'] = load_csv_safe('healthcare_platforms')  # Load healthcare platform presence.
    data['awards'] = load_csv_safe('awards')  # Load awards.
    data['press'] = load_csv_safe('press')  # Load press mentions.
    data['events'] = load_csv_safe('events')  # Load events.
    data['associations'] = load_csv_safe('associations')  # Load professional associations.

    return data  # Return dict to main().

# =====================================================
# DEBUG FUNCTION
# =====================================================

def debug_digital_data(data):  # Debugging helper to inspect digital/social dataset structure; called from main() after load_all_data().
    """Debug function to inspect digital data structure"""
    print("\n" + "="*70)  # Visual separator in logs.
    print("🔍 DIGITAL DATA DEBUG")  # Header for debug section.
    print("="*70)

    if data['digital'].empty:
        print("❌ Digital data is EMPTY!")  # If digital DF empty, log and exit debug.
        return

    print(f"\n📊 Digital.csv Structure:")  # Log structure.
    print(f"   Total rows: {len(data['digital'])}")  # Show number of rows in digital DF.
    print(f"   Unique UINs: {data['digital']['UIN'].nunique()}")  # Unique UIN count assumes 'UIN' column present.
    print(f"   Columns: {data['digital'].columns.tolist()}")  # Print column list.

    print(f"\n📋 Sample Data (first 20 rows):")  # Print sample rows for inspection.
    print(data['digital'].head(20).to_string())  # Print first 20 rows as string.

    # Check for platform-related columns
    print(f"\n🔍 Looking for platform columns:")  # Header.
    platform_cols = [col for col in data['digital'].columns if 'platform' in col.lower() or 'channel' in col.lower()]  # Search for platform/channel columns by name.
    print(f"   Found: {platform_cols}")  # Log found platform columns.

    if platform_cols:
        for col in platform_cols:
            print(f"\n   Column '{col}' - unique values:")  # For each platform-like column, print top value counts.
            print(f"      {data['digital'][col].value_counts().head(10)}")  # Print top 10 unique values.

    # Check sm_channel specifically
    if 'sm_channel' in data['digital'].columns:
        print(f"\n📱 sm_channel distribution:")  # If specific column 'sm_channel' exists, show distribution counts.
        print(data['digital']['sm_channel'].value_counts())  # Print counts.

        # Sample UINs with their channels
        print(f"\n   Sample UINs with channels:")  # Print sample mapping of UIN->sm_channel.
        sample = data['digital'][['UIN', 'sm_channel']].head(30)  # Grab first 30 rows for sample.
        print(sample.to_string())  # Print sample to string for console readability.

# =====================================================
# STEP 2: DATA AGGREGATION (FIXED!)
# =====================================================

def aggregate_data_by_uin(data):  # Aggregate multiple datasets by UIN into a master 'merged' DataFrame; used by main() after load_all_data().
    """Aggregate all data sources by UIN with robust handling"""
    print("\n🔗 Aggregating Data by UIN...")  # Header.

    # CRITICAL FIX: Use PI as master (10,000 doctors), not engagement (may have more/less)
    if data['pi'].empty:
        print("   ❌ No PI data available - cannot proceed")  # If PI is empty, cannot build master list — exit with empty DF.
        return pd.DataFrame()

    master = data['pi'][['UIN']].drop_duplicates().copy()  # master DataFrame built from PI's unique UINs; used as baseline for merges.
    print(f"   Master list: {len(master)} UINs from PI data (all doctors)")  # Log master size.

    if not data['engagement'].empty:
        print(f"   Engagement data available for: {len(data['engagement'])} doctors")  # Log engagement count if present.

    # Initialize all count columns with 0
    master['publication_count'] = 0  # initialize column for publications counts; later overwritten if publications DF exists.
    master['trial_count'] = 0  # initialize trial count.
    master['academic_association_count'] = 0  # initialize academic association count.
    master['award_count'] = 0  # initialize award count.
    master['platform_count'] = 0  # initialize social platform count.
    master['total_followers'] = 0  # initialize sum of followers.
    master['healthcare_platform_count'] = 0  # initialize healthcare-platform presence count.
    master['press_count'] = 0  # initialize press mention count.
    master['event_count'] = 0  # initialize event count.
    master['association_count'] = 0  # initialize association count.

    # Publications
    if not data['publications'].empty and 'UIN' in data['publications'].columns:
        pubs = data['publications'].groupby('UIN').agg(  # Aggregate publications by UIN.
            publication_count=('UIN', 'count')  # Count rows per UIN -> publication_count.
        ).reset_index()
        master = master.merge(pubs[['UIN', 'publication_count']], on='UIN', how='left', suffixes=('', '_new'))  # Left merge to preserve master rows.
        master['publication_count'] = master['publication_count_new'].fillna(0)  # Fill NaN with 0 if no pubs for UIN.
        master = master.drop(columns=['publication_count_new'], errors='ignore')  # Drop intermediate column if exists.
        print(f"   ✓ Merged publications")  # Log success.

    # Trials
    if not data['trials'].empty and 'UIN' in data['trials'].columns:
        trials = data['trials'].groupby('UIN').agg(  # Aggregate trials by UIN.
            trial_count=('UIN', 'count')  # Count rows -> trial_count.
        ).reset_index()
        master = master.merge(trials[['UIN', 'trial_count']], on='UIN', how='left', suffixes=('', '_new'))  # Merge counts into master.
        master['trial_count'] = master['trial_count_new'].fillna(0)  # Replace NaN with 0.
        master = master.drop(columns=['trial_count_new'], errors='ignore')  # Drop temp col.
        print(f"   ✓ Merged trials")  # Log.

    # Academic Associations
    if not data['academic_associations'].empty and 'UIN' in data['academic_associations'].columns:
        acad = data['academic_associations'].groupby('UIN').agg(  # Aggregate academic associations by UIN.
            academic_association_count=('UIN', 'count')  # Count rows -> academic_association_count.
        ).reset_index()
        master = master.merge(acad[['UIN', 'academic_association_count']], on='UIN', how='left', suffixes=('', '_new'))  # Merge counts.
        master['academic_association_count'] = master['academic_association_count_new'].fillna(0)  # Fill NaN with 0.
        master = master.drop(columns=['academic_association_count_new'], errors='ignore')  # Drop temp col.
        print(f"   ✓ Merged academic associations")  # Log.

    # Awards
    if not data['awards'].empty and 'UIN' in data['awards'].columns:
        awards = data['awards'].groupby('UIN').agg(  # Aggregate awards by UIN.
            award_count=('UIN', 'count')  # Count -> award_count.
        ).reset_index()
        master = master.merge(awards[['UIN', 'award_count']], on='UIN', how='left', suffixes=('', '_new'))  # Merge awards counts.
        master['award_count'] = master['award_count_new'].fillna(0)  # Replace NaN.
        master = master.drop(columns=['award_count_new'], errors='ignore')  # Drop intermediate.
        print(f"   ✓ Merged awards")  # Log.

    # *** FIXED: Digital Presence (PROPER INDENTATION!) ***
    if not data['digital'].empty and 'UIN' in data['digital'].columns:  # Only proceed if digital DF exists and has UIN.
        print(f"\n   🔍 DEBUG - Digital Data Analysis:")  # Debug header.
        print(f"      Total rows in digital.csv: {len(data['digital'])}")  # Log rows count.
        print(f"      Columns: {data['digital'].columns.tolist()}")  # Log columns list.
        print(f"      First 5 UINs: {data['digital']['UIN'].head().tolist()}")  # Print sample UINs.
        print(f"      Non-null UINs: {data['digital']['UIN'].notna().sum()}")  # Count non-null UINs.
        print(f"      Non-empty UINs: {(data['digital']['UIN'] != '').sum()}")  # Count non-empty (non-empty-string) UINs.

        # CRITICAL FIX: Filter out blank/empty UINs AND blank sm_channel
        digital_clean = data['digital'][  # Create cleaned subset 'digital_clean' filtering invalid UINs and missing sm_channel.
            (data['digital']['UIN'].notna()) & 
            (data['digital']['UIN'] != '') &
            (data['digital']['sm_channel'].notna())  # Require sm_channel present (assumes column exists in raw digital).
        ].copy()
        print(f"      After cleaning (UIN + sm_channel): {len(digital_clean)} valid rows")  # Log cleaned rows.
        print(f"      Rows with actual platforms: {len(digital_clean)}")  # Same as above for clarity.

        if len(digital_clean) > 0:
            print(f"      Sample cleaned UINs: {digital_clean['UIN'].head(10).tolist()}")  # Print sample UINs.

            agg_dict = {'platform_count': ('UIN', 'count')}  # agg_dict to compute platform_count = number of rows per UIN (platform occurrences).

            # Check for followers column
            if 'sm_followers' in digital_clean.columns:
                print(f"      sm_followers column found!")  # Log if follower counts present.
                print(f"      Sample followers: {digital_clean['sm_followers'].head(10).tolist()}")  # Show sample follower values.
                # Convert to numeric, handling any non-numeric values
                digital_clean['sm_followers'] = pd.to_numeric(digital_clean['sm_followers'], errors='coerce').fillna(0)  # Normalize followers to numeric, default 0.
                agg_dict['total_followers'] = ('sm_followers', 'sum')  # Add aggregation for total_followers (sum of sm_followers per UIN).
            else:
                print(f"      ⚠ sm_followers column NOT found")  # Warn if follower column missing.

            digital = digital_clean.groupby('UIN').agg(**agg_dict).reset_index()  # groupby UIN and apply agg_dict -> 'digital' DF with platform_count and maybe total_followers.
            print(f"      After groupby: {len(digital)} unique UINs")  # Log aggregated UIN count.
            print(f"      Sample aggregated data:")  # Debug print sample of aggregated digital DF.
            print(digital.head(10))  # Show head of digital.

            # Check overlap before merge
            overlap = set(master['UIN']) & set(digital['UIN'])  # Compute set intersection of master and digital UINs for diagnostics.
            print(f"      🔍 UIN overlap: {len(overlap)} doctors in both master and digital")  # Print overlap count.
            print(f"      🔍 Sample overlapping UINs: {list(overlap)[:10]}")  # Show sample overlaps.

            # Debug BEFORE merge
            print(f"      🔍 BEFORE MERGE - master['platform_count'].sum(): {master['platform_count'].sum():.0f}")  # Current sum in master (should be 0 initially).
            print(f"      🔍 BEFORE MERGE - digital['platform_count'].sum(): {digital['platform_count'].sum():.0f}")  # Sum of platform_count in digital.

            master = master.merge(digital, on='UIN', how='left', suffixes=('_old', ''))  # Merge digital aggregates into master by UIN; left join to keep all master rows.

            # Check if we have duplicate columns
            print(f"      🔍 Columns after merge: {[c for c in master.columns if 'platform' in c or 'follower' in c]}")  # Print any platform/follower-related columns post-merge.

            # If we have _old columns, drop them and keep the new ones
            if 'platform_count_old' in master.columns:
                master = master.drop(columns=['platform_count_old'], errors='ignore')  # Remove older platform_count col if present.
                print(f"      🔍 Dropped old platform_count column")  # Log.

            if 'total_followers_old' in master.columns:
                master = master.drop(columns=['total_followers_old'], errors='ignore')  # Remove old followers col if present.
                print(f"      🔍 Dropped old total_followers column")  # Log.

            if 'platform_count' in master.columns:
                master['platform_count'] = master['platform_count'].fillna(0)  # Ensure platform_count has no NaN; fill zeros for missing.
            else:
                master['platform_count'] = 0  # If missing entirely, create with 0.

            if 'total_followers' in master.columns:
                master['total_followers'] = master['total_followers'].fillna(0)  # Ensure total_followers has no NaN.
            else:
                master['total_followers'] = 0  # Create column if absent.

            print(f"   ✓ Merged digital data: {len(digital)} UINs with social media presence")  # Log merge completion.
            print(f"      ✓ AFTER MERGE - Doctors with platform_count > 0: {(master['platform_count'] > 0).sum()}")  # Count doctors with platforms.
            print(f"      ✓ AFTER MERGE - Doctors with total_followers > 0: {(master['total_followers'] > 0).sum()}")  # Count doctors with followers.
            print(f"      ✓ AFTER MERGE - Total platforms: {master['platform_count'].sum():.0f}")  # Sum of platform_count across master.
            print(f"      ✓ AFTER MERGE - Total followers: {master['total_followers'].sum():.0f}")  # Sum of followers across master.
        else:
            print(f"   ⚠ Digital data has no valid UINs")  # If cleaning removed all rows, log this.
    else:
        print(f"   ⚠ Digital data is empty or missing UIN column")  # No digital DF or missing UIN.

    # Healthcare Platforms
    if not data['healthcare_platforms'].empty and 'UIN' in data['healthcare_platforms'].columns:
        hc_plat = data['healthcare_platforms'].groupby('UIN').agg(  # Aggregate healthcare platform presence by UIN.
            healthcare_platform_count=('UIN', 'count')  # Count -> healthcare_platform_count.
        ).reset_index()
        master = master.merge(hc_plat[['UIN', 'healthcare_platform_count']], on='UIN', how='left', suffixes=('', '_new'))  # Merge into master.
        master['healthcare_platform_count'] = master['healthcare_platform_count_new'].fillna(0)  # Fill NaN with 0.
        master = master.drop(columns=['healthcare_platform_count_new'], errors='ignore')  # Drop intermediate.
        print(f"   ✓ Merged healthcare platforms")  # Log.

    # Press
    if not data['press'].empty and 'UIN' in data['press'].columns:
        press = data['press'].groupby('UIN').agg(  # Aggregate press mentions by UIN.
            press_count=('UIN', 'count')  # Count -> press_count.
        ).reset_index()
        master = master.merge(press[['UIN', 'press_count']], on='UIN', how='left', suffixes=('', '_new'))  # Merge into master.
        master['press_count'] = master['press_count_new'].fillna(0)  # Fill NaN.
        master = master.drop(columns=['press_count_new'], errors='ignore')  # Drop temp.
        print(f"   ✓ Merged press")  # Log.

    # Events
    if not data['events'].empty and 'UIN' in data['events'].columns:
        events = data['events'].groupby('UIN').agg(  # Aggregate event participations by UIN.
            event_count=('UIN', 'count')  # Count -> event_count.
        ).reset_index()
        master = master.merge(events[['UIN', 'event_count']], on='UIN', how='left', suffixes=('', '_new'))  # Merge counts.
        master['event_count'] = master['event_count_new'].fillna(0)  # Fill NaN.
        master = master.drop(columns=['event_count_new'], errors='ignore')  # Drop intermediate col.
        print(f"   ✓ Merged events")  # Log.

    # Associations
    if not data['associations'].empty and 'UIN' in data['associations'].columns:
        assoc = data['associations'].groupby('UIN').agg(  # Aggregate association membership by UIN.
            association_count=('UIN', 'count')  # Count -> association_count.
        ).reset_index()
        master = master.merge(assoc[['UIN', 'association_count']], on='UIN', how='left', suffixes=('', '_new'))  # Merge into master.
        master['association_count'] = master['association_count_new'].fillna(0)  # Fill NaN.
        master = master.drop(columns=['association_count_new'], errors='ignore')  # Drop temp.
        print(f"   ✓ Merged associations")  # Log.

    master = master.fillna(0)  # Fill any remaining NaNs across master with 0 (safety net).

    # Add engagement data (left join - not all doctors may have engagement)
    if not data['engagement'].empty:
        merged = master.merge(data['engagement'], on='UIN', how='left')  # Left join engagement averages onto master (keeps all master UINs).
        print(f"   ✓ Merged engagement data")  # Log.
    else:
        merged = master.copy()  # Fall back: copy master to merged if no engagement DF.
        # Add default engagement columns
        for col in ['hcp_email_open_rate', 'hcp_email_click_rate', 'hcp_whatsapp_read_rate',
                   'hcp_whatsapp_click_rate', 'hcp_call_productive_rate', 'average_duration_connected_calls']:
            merged[col] = 0  # Create engagement columns with default 0 when no engagement data is present.

    # Add PI data
    if not data['pi'].empty and 'UIN' in data['pi'].columns:
        available_cols = ['UIN']  # We'll collect optional PI columns to merge (name, mobile, specialty etc.)
        for col in ['full_name', 'mobile', 'whatsapp_phone', 'email_id_1', 'specialty']:
            if col in data['pi'].columns:
                available_cols.append(col)  # Append column if present in PI DF.

        if len(available_cols) > 1:  # If we found any additional columns besides 'UIN'
            pi_subset = data['pi'][available_cols].drop_duplicates('UIN')  # Prepare unique subset keyed by UIN.
            merged = merged.merge(pi_subset, on='UIN', how='left')  # Merge PI details into merged DF.
            print(f"   ✓ Merged PI data")  # Log.

    # Add clinic data
    if not data['clinics'].empty and 'UIN' in data['clinics'].columns:
        available_cols = ['UIN']  # Collect optional clinic columns if present.
        for col in ['clinic_address', 'clinic_city', 'clinic_state']:
            if col in data['clinics'].columns:
                available_cols.append(col)  # Append if exists.

        if len(available_cols) > 1:  # If any clinic columns present
            clinic_subset = data['clinics'][available_cols].drop_duplicates('UIN')  # Prepare unique clinics subset.
            merged = merged.merge(clinic_subset, on='UIN', how='left')  # Merge into merged DF.
            print(f"   ✓ Merged clinic data")  # Log.

    print(f"   ✓ Final merged dataset: {len(merged)} UINs, {len(merged.columns)} columns")  # Final merged DF diagnostics.

    # Ensure all expected columns exist
    for col in ['publication_count', 'trial_count', 'academic_association_count',
                'award_count', 'platform_count', 'total_followers', 
                'healthcare_platform_count', 'press_count', 'event_count',
                'association_count']:
        if col not in merged.columns:
            merged[col] = 0  # Guarantee presence of expected metrics columns to avoid KeyError later.

    return merged  # Return merged DataFrame to caller (main()).


# =====================================================
# STEP 3: SCORING - UPDATED TO USE SCORING_CONFIG
# =====================================================

def calculate_engagement_score(row, config):  # Calculates engagement_score per doctor row; called inside score_all_doctors() via apply().
    """Calculate engagement bucket score"""
    email_open = row.get('hcp_email_open_rate', 0) or 0  # Read email open rate (from merged DF) or default 0.
    email_click = row.get('hcp_email_click_rate', 0) or 0  # Read email click rate or default 0.
    email_score = (email_open * config['email_open_weight'] + 
                   email_click * config['email_click_weight'])  # Compose email subscore using weights from SCORING_CONFIG['engagement'].

    wa_read = row.get('hcp_whatsapp_read_rate', 0) or 0  # WhatsApp read rate (merged DF column).
    wa_click = row.get('hcp_whatsapp_click_rate', 0) or 0  # WhatsApp click rate.
    wa_score = (wa_read * config['whatsapp_read_weight'] + 
                wa_click * config['whatsapp_click_weight'])  # Compose WhatsApp subscore.

    call_prod = row.get('hcp_call_productive_rate', 0) or 0  # Productive call rate (merged DF).
    call_dur = row.get('average_duration_connected_calls', 0) or 0  # Average call duration (seconds perhaps) (merged DF).
    call_dur_norm = min((call_dur / 60) * 100, 100) if call_dur > 0 else 0  # Normalize duration to percentage (cap at 100).
    call_score = (call_prod * config['call_productive_weight'] + 
                  call_dur_norm * config['call_duration_weight'])  # Compose call subscore with weights.

    final = (email_score * config['final_email_weight'] + 
             wa_score * config['final_whatsapp_weight'] + 
             call_score * config['final_call_weight'])  # Weighted final engagement combining email, whatsapp, calls.

    return round(final, 2)  # Return engagement score rounded to 2 decimals.


def calculate_academic_score(row, config):  # Compute academic_score per row using counts in merged DF; used in score_all_doctors().
    """Calculate academic/research bucket score"""
    pubs = row.get('publication_count', 0) or 0  # Number of publications (merged DF).
    trials = row.get('trial_count', 0) or 0  # Number of trials (merged DF).
    acad_assoc = row.get('academic_association_count', 0) or 0  # Academic associations count.

    pub_score = min(pubs * config['publication_points_per_item'], config['publication_max_score'])  # Each publication gives points capped at max.
    trial_score = min(trials * config['trial_points_per_item'], config['trial_max_score'])  # Each trial gives points capped at max.
    assoc_score = min(acad_assoc * config['association_points_per_item'], config['association_max_score'])  # Each academic assoc gives points capped at max.

    total = pub_score + trial_score + assoc_score  # Sum components.
    return round(min(total, config['max_score']), 2)  # Cap at max_score and round.


def calculate_social_media_score(row, config):  # Compute social_media_score using platform and follower info; used in score_all_doctors().
    """Calculate social media inclination bucket score"""
    platforms = row.get('platform_count', 0) or 0  # Number of platforms (merged DF).
    followers = row.get('total_followers', 0) or 0  # Total followers (merged DF).
    hc_platforms = row.get('healthcare_platform_count', 0) or 0  # Healthcare-specific platform count.

    if followers > 0:
        follower_score = min(np.log10(followers) * config['follower_log_multiplier'], config['follower_max_score'])  # Use log10 of followers scaled to max to avoid skew.
    else:
        follower_score = 0  # No followers -> 0.

    platform_score = min(platforms * config['platform_points_per_item'], config['platform_max_score'])  # Each platform gives points capped at max.
    hc_score = min(hc_platforms * config['healthcare_platform_points_per_item'], config['healthcare_platform_max_score'])  # Healthcare platform score capped at max.

    total = follower_score + platform_score + hc_score  # Sum components.
    return round(min(total, config['max_score']), 2)  # Cap and round.


def calculate_recognition_score(row, config):  # Compute recognition_score using awards, press, events, associations; used in score_all_doctors().
    """Calculate recognition bucket score"""
    awards = row.get('award_count', 0) or 0  # Awards count.
    press = row.get('press_count', 0) or 0  # Press mentions count.
    events = row.get('event_count', 0) or 0  # Events count.
    assoc = row.get('association_count', 0) or 0  # Associations count.

    award_score = min(awards * config['award_points_per_item'], config['award_max_score'])  # Awards heavy weight, capped.
    press_score = min(press * config['press_points_per_item'], config['press_max_score'])  # Press capped.
    event_score = min(events * config['event_points_per_item'], config['event_max_score'])  # Events capped.
    assoc_score = min(assoc * config['association_points_per_item'], config['association_max_score'])  # Associations capped.

    total = award_score + press_score + event_score + assoc_score  # Sum.
    return round(min(total, config['max_score']), 2)  # Cap and round.


def score_all_doctors(merged_data):  # Apply scoring functions across merged dataset to generate bucket scores per doctor; called in main().
    """Calculate all 4 bucket scores for each doctor"""
    print("\n🎯 Calculating Scores for All 4 Buckets...")  # Log.

    df = merged_data.copy()  # Work on a copy to avoid mutating original merged DataFrame.

    # Calculate scores using configuration for each bucket
    df['engagement_score'] = df.apply(lambda row: calculate_engagement_score(row, SCORING_CONFIG['engagement']), axis=1)  # Compute engagement_score using apply.
    df['academic_score'] = df.apply(lambda row: calculate_academic_score(row, SCORING_CONFIG['academic']), axis=1)  # Compute academic_score via apply.
    df['social_media_score'] = df.apply(lambda row: calculate_social_media_score(row, SCORING_CONFIG['social_media']), axis=1)  # Compute social_media_score.
    df['recognition_score'] = df.apply(lambda row: calculate_recognition_score(row, SCORING_CONFIG['recognition']), axis=1)  # Compute recognition_score.

    df['engagement_data_available'] = df['engagement_score'] > 0  # Boolean flag if engagement_score > 0.
    df['academic_data_available'] = (  # Boolean if any academic-related counts > 0.
        df.get('publication_count', 0).fillna(0) + 
        df.get('trial_count', 0).fillna(0) + 
        df.get('academic_association_count', 0).fillna(0)
    ) > 0
    df['social_media_data_available'] = (  # Boolean if platform or healthcare platform counts > 0.
        df.get('platform_count', 0).fillna(0) + 
        df.get('healthcare_platform_count', 0).fillna(0)
    ) > 0
    df['recognition_data_available'] = (  # Boolean if any recognition metrics > 0.
        df.get('award_count', 0).fillna(0) + 
        df.get('press_count', 0).fillna(0) + 
        df.get('event_count', 0).fillna(0) + 
        df.get('association_count', 0).fillna(0)
    ) > 0

    df['buckets_with_data'] = (  # Numerical count of how many buckets have data (0..4).
        df['engagement_data_available'].astype(int) +
        df['academic_data_available'].astype(int) +
        df['social_media_data_available'].astype(int) +
        df['recognition_data_available'].astype(int)
    )

    print(f"   ✓ Scored {len(df)} doctors")  # Diagnostic: total rows scored.
    print(f"   Average scores:")  # Print average scores for inspection.
    print(f"      Engagement: {df['engagement_score'].mean():.2f}")  # Mean engagement_score.
    print(f"      Academic: {df['academic_score'].mean():.2f}")  # Mean academic_score.
    print(f"      Social Media: {df['social_media_score'].mean():.2f}")  # Mean social_media_score.
    print(f"      Recognition: {df['recognition_score'].mean():.2f}")  # Mean recognition_score.
    print(f"\n   Data Availability:")  # Print data availability metrics.
    print(f"      Doctors with social media data: {df['social_media_data_available'].sum()}")  # Count with social data.
    print(f"      Total platforms tracked: {df['platform_count'].sum():.0f}")  # Sum platform_count across dataset.
    print(f"      Total followers tracked: {df['total_followers'].sum():.0f}")  # Sum followers.

    return df  # Return scored DataFrame to main().

# =====================================================
# STEP 4: CLUSTERING ELIGIBILITY
# =====================================================

def assess_clustering_eligibility(df):  # Determine which doctors meet eligibility rules for clustering; called in main() after scoring.
    """Determine which doctors are eligible for clustering"""
    print("\n✅ Assessing Clustering Eligibility...")  # Log.

    rules = ELIGIBILITY_RULES[ELIGIBILITY_MODE]  # Select rule set based on ELIGIBILITY_MODE from CONFIG (e.g., strict/moderate).

    df['eligible_for_clustering'] = (  # Boolean column: True if doctor meets min_buckets and optionally engagement requirement.
        (df['buckets_with_data'] >= rules['min_buckets']) &
        (df['engagement_data_available'] if rules['engagement_required'] else True)
    )

    def get_ineligibility_reason(row):  # Helper to compute a human-readable reason for ineligibility for each row; applied below.
        if row['eligible_for_clustering']:
            return ''  # Empty string if eligible.
        reasons = []  # Build list of reasons why ineligible.
        if not row['engagement_data_available']:
            reasons.append('No engagement data')  # Add engagement reason if missing.
        if row['buckets_with_data'] < rules['min_buckets']:
            reasons.append(f"Only {int(row['buckets_with_data'])}/{rules['min_buckets']} buckets have data")  # Add bucket-count reason.
        return '; '.join(reasons)  # Join reasons with semicolon.

    df['insufficient_data_reason'] = df.apply(get_ineligibility_reason, axis=1)  # Create explanatory column via apply.

    eligible_count = df['eligible_for_clustering'].sum()  # Number of rows eligible.
    ineligible_count = len(df) - eligible_count  # Number ineligible for clustering.

    print(f"   Mode: {ELIGIBILITY_MODE}")  # Log mode used.
    print(f"   ✓ Eligible for clustering: {eligible_count} doctors")  # Log counts.
    print(f"   ✗ Ineligible: {ineligible_count} doctors")  # Log counts.

    print(f"\n   Bucket Data Availability:")  # Print breakdown of which buckets have data counts.
    print(f"      Engagement: {df['engagement_data_available'].sum()} doctors")  # Count with engagement.
    print(f"      Academic: {df['academic_data_available'].sum()} doctors")  # Count with academic data.
    print(f"      Social Media: {df['social_media_data_available'].sum()} doctors")  # Count with social data.
    print(f"      Recognition: {df['recognition_data_available'].sum()} doctors")  # Count with recognition.

    return df  # Return DataFrame augmented with eligibility columns.

# =====================================================
# STEP 5: K-MEANS CLUSTERING
# =====================================================

def perform_clustering(df, n_clusters=5):  # Perform KMeans clustering on eligible doctors. Called in main() with n_clusters=5 by default.
    """Perform K-Means clustering on eligible doctors"""
    print(f"\n🔍 Performing K-Means Clustering (k={n_clusters})...")  # Log.

    eligible = df[df['eligible_for_clustering']].copy()  # Subset DataFrame to only eligible doctors for clustering.

    if len(eligible) < n_clusters:  # If not enough samples for requested clusters:
        print(f"   ❌ Not enough eligible doctors ({len(eligible)}) for {n_clusters} clusters")  # Log.
        df['cluster_id'] = None  # Ensure cluster_id column exists as None for all rows.
        df['cluster_name'] = None  # Ensure cluster_name exists as None.
        return df, []  # Return original DataFrame and empty cluster_profiles list.

    # CRITICAL FIX: Fill NaN values with 0 before clustering
    score_cols = ['engagement_score', 'academic_score', 'social_media_score', 'recognition_score']  # Columns used as features.
    for col in score_cols:
        eligible[col] = eligible[col].fillna(0)  # Replace NaN in feature columns with 0 to avoid issues in scaler/KMeans.

    features = eligible[score_cols].values  # Extract numpy array of features (shape: n_samples x 4).

    # Check for any remaining NaN values
    if np.isnan(features).any():
        print(f"   ⚠ WARNING: NaN values detected in features, replacing with 0")  # If any NaN remain, notify.
        features = np.nan_to_num(features, nan=0.0)  # Replace NaNs with 0.0 in NumPy array.

    # ✅ SCALE FIX: Standardize features so all dimensions have equal weight
    print(f"\n   📊 Feature statistics BEFORE scaling:")  # Print pre-scaling stats for diagnostics.
    print(f"      Engagement: mean={eligible['engagement_score'].mean():.2f}, std={eligible['engagement_score'].std():.2f}")  # Engagement stats.
    print(f"      Academic: mean={eligible['academic_score'].mean():.2f}, std={eligible['academic_score'].std():.2f}")  # Academic stats.
    print(f"      Social Media: mean={eligible['social_media_score'].mean():.2f}, std={eligible['social_media_score'].std():.2f}")  # Social media stats.
    print(f"      Recognition: mean={eligible['recognition_score'].mean():.2f}, std={eligible['recognition_score'].std():.2f}")  # Recognition stats.

    scaler = StandardScaler()  # Instantiate StandardScaler (from sklearn) — used to normalize features.
    features_scaled = scaler.fit_transform(features)  # Fit scaler and transform feature matrix -> features_scaled used for KMeans.

    print(f"\n   📊 Feature statistics AFTER scaling (all should be ~0 mean, ~1 std):")  # Post-scale diagnostics.
    print(f"      Engagement: mean={features_scaled[:,0].mean():.2f}, std={features_scaled[:,0].std():.2f}")  # Engagement scaled stats.
    print(f"      Academic: mean={features_scaled[:,1].mean():.2f}, std={features_scaled[:,1].std():.2f}")  # Academic scaled stats.
    print(f"      Social Media: mean={features_scaled[:,2].mean():.2f}, std={features_scaled[:,2].std():.2f}")  # Social scaled stats.
    print(f"      Recognition: mean={features_scaled[:,3].mean():.2f}, std={features_scaled[:,3].std():.2f}")  # Recognition scaled stats.

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)  # Configure KMeans with reproducible random_state & multiple inits.
    clusters = kmeans.fit_predict(features_scaled)  # Fit KMeans and predict cluster labels for each eligible doctor.

    if len(eligible) > n_clusters:
        sil_score = silhouette_score(features_scaled, clusters)  # Compute silhouette score if there are enough samples.
        print(f"\n   ✓ Clustering completed")  # Log completion.
        print(f"   Silhouette Score: {sil_score:.3f}")  # Print silhouette score for cluster quality.
    else:
        print(f"\n   ✓ Clustering completed (too few samples for silhouette score)")  # If insufficient samples, skip silhouette.

    eligible['cluster_id'] = clusters  # Attach cluster_id labels to eligible subset.

    # Get cluster centroids in scaled space
    centroids = kmeans.cluster_centers_  # Centroids in the scaled feature space (n_clusters x 4 matrix).

    cluster_profiles = []  # Will collect per-cluster summary dictionaries.
    for i in range(n_clusters):  # Iterate all cluster indices (0..n_clusters-1).
        cluster_data = eligible[eligible['cluster_id'] == i]  # Subset eligible rows belonging to cluster i.
        profile = {
            'cluster_id': i,  # cluster index
            'size': len(cluster_data),  # number of doctors in this cluster
            'avg_engagement': cluster_data['engagement_score'].mean(),  # mean raw engagement_score for cluster members
            'avg_academic': cluster_data['academic_score'].mean(),  # mean academic_score
            'avg_social_media': cluster_data['social_media_score'].mean(),  # mean social_media_score
            'avg_recognition': cluster_data['recognition_score'].mean()  # mean recognition_score
            }

        # ✅ FIX: Use the scaled centroid values to determine dominance
        centroid = centroids[i]  # Scaled centroid for cluster i (array of length 4).
        traits = ['Engagement', 'Academic', 'Social Media', 'Recognition']  # Friendly labels corresponding to score_cols order.
        centroid_dict = {traits[j]: centroid[j] for j in range(4)}  # Map labels to scaled centroid values for interpretability.

        # Find which dimension has the highest value in scaled space
        dominant = max(centroid_dict, key=centroid_dict.get)  # Determine dominant trait by highest scaled centroid value.
        profile['cluster_name'] = f"Cluster {i}: {dominant}-Focused"  # Compose human-friendly cluster name.
        profile['scaled_centroid'] = centroid_dict  # Store centroid dict for detailed profile (may be saved later).

        cluster_profiles.append(profile)  # Append profile to list returned to caller.
        print(f"\n   Cluster {i}: {len(cluster_data)} doctors - {profile['cluster_name']}")  # Log cluster summary.
        print(f"      Raw scores: Eng={profile['avg_engagement']:.1f}, Acad={profile['avg_academic']:.1f}, SM={profile['avg_social_media']:.1f}, Recog={profile['avg_recognition']:.1f}")  # Print mean raw scores.
        print(f"      Scaled centroids: Eng={centroid_dict['Engagement']:.2f}, Acad={centroid_dict['Academic']:.2f}, SM={centroid_dict['Social Media']:.2f}, Recog={centroid_dict['Recognition']:.2f}")  # Print scaled centroid values.

    # Merge clusters back to main dataframe
    df = df.merge(eligible[['UIN', 'cluster_id']], on='UIN', how='left', suffixes=('', '_new'))  # Left-merge cluster_id from eligible subset back onto full df.
    if 'cluster_id_new' in df.columns:
        df['cluster_id'] = df['cluster_id_new']  # If suffix created cluster_id_new, move it into cluster_id column.
        df.drop('cluster_id_new', axis=1, inplace=True)  # Drop temporary column to keep schema clean.

    # Add cluster names
    cluster_map = {p['cluster_id']: p['cluster_name'] for p in cluster_profiles}  # Map cluster_id -> cluster_name from cluster_profiles.
    df['cluster_name'] = df['cluster_id'].map(cluster_map)  # Map names to df's cluster_id column (NaN remain for non-clustered rows).

    return df, cluster_profiles  # Return augmented DataFrame (with cluster_id/name) and cluster_profiles list to caller (main()).

# =====================================================
# STEP 6: OUTPUT GENERATION
# =====================================================

def generate_output(df, cluster_profiles):  # Writes final CSV outputs and prints summary statistics; called in main() last.
    """Generate final output files - INCLUDES ALL DOCTORS"""
    print("\n💾 Generating Output Files...")  # Log.

    base_cols = ['UIN']  # Always include UIN as primary key in outputs.
    optional_cols = [  # Optional PI/clinic contact columns — added to output if present in df.
        'full_name', 'specialty', 'mobile', 'whatsapp_phone', 'email_id_1',
        'clinic_address', 'clinic_city', 'clinic_state'
    ]
    required_cols = [  # Required metrics and meta columns for the output CSV.
        'engagement_score', 'engagement_data_available',
        'academic_score', 'academic_data_available',
        'social_media_score', 'social_media_data_available',
        'recognition_score', 'recognition_data_available',
        'buckets_with_data', 'eligible_for_clustering', 
        'cluster_id', 'cluster_name', 'insufficient_data_reason'
    ]

    output_cols = base_cols.copy()  # Start building output column list with base columns.

    for col in optional_cols:
        if col in df.columns:
            output_cols.append(col)  # Add optional columns that exist in df.

    output_cols.extend(required_cols)  # Append required metric columns.

    for col in output_cols:
        if col not in df.columns:
            df[col] = None  # Ensure all output columns exist on df (fill with None if missing).

    output_df = df[output_cols].copy()  # Build final output_df with chosen columns in order.

    # Save main output
    output_file = f"{DATA_DIR}\\scored_doctors_with_clusters.csv"  # Path for main CSV output.
    try:
        output_df.to_csv(output_file, index=False, encoding='utf-8')  # Attempt save with utf-8 encoding.
        print(f"   ✓ Saved: {output_file}")  # Success log.
    except Exception as e:
        print(f"   ❌ Error saving main file: {e}")  # Print error if fails.
        try:
            output_df.to_csv(output_file, index=False, encoding='latin-1')  # Fallback save with latin-1 encoding.
            print(f"   ✓ Saved with latin-1 encoding: {output_file}")  # Success with fallback encoding.
        except Exception as e2:
            print(f"   ❌ Could not save main file: {e2}")  # If fallback fails, log error.

    # Save cluster profiles - simple version without scaled centroids
    if cluster_profiles:
        # Create clean dataframe with only the essential columns
        profiles_data = []  # Prepare list of dicts for profiles CSV.
        for profile in cluster_profiles:
            profiles_data.append({  # Only include user-friendly summary fields (omit scaled centroid).
                'cluster_id': profile['cluster_id'],
                'size': profile['size'],
                'avg_engagement': profile['avg_engagement'],
                'avg_academic': profile['avg_academic'],
                'avg_social_media': profile['avg_social_media'],
                'avg_recognition': profile['avg_recognition'],
                'cluster_name': profile['cluster_name']
            })

        profiles_df = pd.DataFrame(profiles_data)  # Convert to DataFrame for saving.

        profiles_file = f"{DATA_DIR}\\cluster_profiles.csv"  # Path for cluster profiles CSV.
        try:
            profiles_df.to_csv(profiles_file, index=False)  # Save CSV.
            print(f"   ✓ Saved: {profiles_file}")  # Log success.
        except Exception as e:
            print(f"   ❌ Error saving profiles: {e}")  # Log any save error.

    # Summary statistics
    print(f"\n📊 Summary Statistics:")  # Overall summary header.
    print(f"   Total doctors in output: {len(output_df)}")  # Print total doctors included in output_df.
    print(f"   Eligible for clustering: {output_df['eligible_for_clustering'].sum()}")  # Count eligible.
    print(f"   Clustered: {output_df['cluster_id'].notna().sum()}")  # Count clustered (non-null cluster_id).
    print(f"   Ineligible (insufficient data): {(~output_df['eligible_for_clustering']).sum()}")  # Count ineligible.

    # Breakdown by buckets
    print(f"\n   Bucket Distribution:")  # Print distribution stats for buckets_with_data.
    for i in range(5):
        count = (output_df['buckets_with_data'] == i).sum()  # Count doctors with exactly i buckets.
        print(f"      {i} buckets: {count} doctors")  # Print.

    return output_df  # Return output_df to caller (main()).

# =====================================================
# MAIN EXECUTION
# =====================================================

def main():  # Entry point for script execution; ties together all steps in order.
    print("=" * 70)  # Decorative header.
    print("HEALTHCARE PROFESSIONAL ANALYTICS PIPELINE")  # Title print.
    print("4-Bucket Scoring + K-Means Clustering")  # Subtitle.
    print("Version 1.3 - With debug function")  # Version info.
    print("=" * 70)  # Decorative.

    try:
        # Step 1: Load data
        data = load_all_data()  # Call load_all_data() to read all CSVs into data dict.

        # Debug digital data
        debug_digital_data(data)  # Print diagnostics for digital.csv to help troubleshoot social media input.

        if data['engagement'].empty:
            print("\n❌ No engagement data loaded - cannot proceed with analysis")  # Stop early if engagement data missing.
            return  # Exit main() gracefully.

        # Step 2: Aggregate by UIN
        merged = aggregate_data_by_uin(data)  # Aggregate all sources into merged DataFrame keyed by UIN.

        if merged.empty:
            print("\n❌ No data to process after aggregation")  # Stop if aggregation failed or produced empty DF.
            return

        # Step 3: Calculate scores
        scored = score_all_doctors(merged)  # Compute 4 bucket scores and availability flags -> 'scored' DF.

        # Step 4: Assess eligibility
        assessed = assess_clustering_eligibility(scored)  # Add eligibility booleans and reasons -> 'assessed' DF.

        # Step 5: Cluster eligible doctors
        clustered, profiles = perform_clustering(assessed, n_clusters=5)  # Run KMeans and get cluster_profiles list.

        # Step 6: Generate output
        final_output = generate_output(clustered, profiles)  # Save CSVs and print final summaries -> output_df returned.

        print("\n" + "=" * 70)  # Success header decoration.
        print("✓ PIPELINE COMPLETED SUCCESSFULLY")  # Success message.
        print("=" * 70)
        print(f"\nOutput files created in: {DATA_DIR}")  # Print output location.
        print("  1. scored_doctors_with_clusters.csv (ALL doctors included)")  # List outputs.
        if profiles:
            print("  2. cluster_profiles.csv")  # Print if cluster_profiles exist.

        print("\n🔍 Key Fixes Applied:")  # Log key fixes summary (human notes).
        print("  ✓ Fixed digital data processing indentation")
        print("  ✓ Filtered out blank UINs in Digital.csv")
        print("  ✓ All 10,000 doctors now included in output")
        print("  ✓ Social media scores now calculated correctly")

    except Exception as e:
        print(f"\n❌ Pipeline failed with error: {e}")  # Catch-all error logging to assist debugging.
        import traceback  # Import traceback in exception path for detailed trace.
        traceback.print_exc()  # Print full stack trace for debugging.

if __name__ == "__main__":
    main()  # If script executed directly, call main() to run the full pipeline.


HEALTHCARE PROFESSIONAL ANALYTICS PIPELINE
4-Bucket Scoring + K-Means Clustering
Version 1.3 - With debug function

📁 Loading All Data Files...

📊 Loading Engagement Data (Jan-July 2025)...
   Loading Jan25.csv...
      Detected encoding: ascii (confidence: 1.00)
      Successfully loaded with ascii encoding
   ✓ Loaded Jan25.csv: 1048575 rows, 11570 unique UINs
   Loading Feb25.csv...
      Detected encoding: ascii (confidence: 1.00)
      Successfully loaded with ascii encoding
   ✓ Loaded Feb25.csv: 12954 rows, 12953 unique UINs
   Loading March25.csv...
      Detected encoding: ascii (confidence: 1.00)
      Successfully loaded with ascii encoding
   ✓ Loaded March25.csv: 13041 rows, 13040 unique UINs
   Loading April25.csv...
      Detected encoding: ascii (confidence: 1.00)
      Successfully loaded with ascii encoding
   ✓ Loaded April25.csv: 13797 rows, 13796 unique UINs
   Loading May25.csv...
      Detected encoding: ascii (confidence: 1.00)
      Successfully loaded with asc